# 資料處理

## 讀nc4

In [2]:
import os
import glob
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm


def check_data_folder(folder: str) -> bool:
    return os.path.exists(folder) and os.path.isdir(folder)


def load_data(file_path: str) -> xr.Dataset:
    """
    Load data from a NetCDF file, trying netcdf4 then h5netcdf.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    # 優先 netcdf4
    try:
        return xr.open_dataset(file_path, engine="netcdf4")
    except Exception as e1:
        # 改試 h5netcdf
        try:
            return xr.open_dataset(file_path, engine="h5netcdf")
        except Exception as e2:
            raise RuntimeError(
                f"Failed to open {file_path} with netcdf4 and h5netcdf.\n"
                f"e1: {e1}\n"
                f"e2: {e2}\n"
                "請確認這個環境有安裝 netCDF4 或 h5netcdf。"
            )


# ================= main program =================

data_folder = "nc4"
var_name = "T2M"  # 目標變數名稱（請確認檔內真的叫這個）

# 1. 檢查資料夾
if not check_data_folder(data_folder):
    raise FileNotFoundError(f"Data folder not found: {data_folder}")
print(f"Data folder found: {data_folder}")

# 2. 找出所有像 1980-01.nc4 的檔案
pattern = os.path.join(data_folder, "*.nc4")
file_list = sorted(glob.glob(pattern))

if len(file_list) == 0:
    raise FileNotFoundError(f"No nc4 files found with pattern: {pattern}")

print(f"Found {len(file_list)} files.")
print("First 5 files:", file_list[:5])

# 3. 用第一個檔案確認經緯度與變數存在，必要時自動偵測 var_name
with load_data(file_list[0]) as sample_data:
    print("Data variables in first file:")
    print(list(sample_data.data_vars))
    print("Coordinates:")
    print({k: sample_data[k].shape for k in sample_data.coords})

    # 找出所有長得像 (time, lat, lon) 的候選變數
    candidates = []
    for v in sample_data.data_vars:
        dims = set(sample_data[v].dims)
        if {"time", "lat", "lon"}.issubset(dims):
            candidates.append(v)

    # 如果原本設定的 var_name 不在，就試著自動改
    if var_name not in sample_data.data_vars:
        if len(candidates) == 1:
            auto_var = candidates[0]
            print(f"[Info] Variable '{var_name}' not found, auto-select '{auto_var}' as target.")
            var_name = auto_var
        else:
            raise KeyError(
                f"Variable '{var_name}' not found in file: {file_list[0]}\n"
                f"Available data_vars: {list(sample_data.data_vars)}\n"
                f"3D (time,lat,lon) candidates: {candidates}\n"
                "請將上方其中一個正確變數名稱填入 var_name。"
            )

    # 取經緯度
    if "lat" not in sample_data.coords or "lon" not in sample_data.coords:
        raise KeyError("lat/lon coordinates not found in sample file.")
    lat = sample_data["lat"].values
    lon = sample_data["lon"].values

nlat = lat.shape[0]
nlon = lon.shape[0]
print(f"Confirmed var_name = {var_name}")
print(f"Lat: {nlat}, Lon: {nlon}")


# 4. 逐檔讀入，累積到 list
data_list = []
time_list = []

for f in tqdm(file_list, desc="Combining"):
    with load_data(f) as ds:
        da = ds[var_name]  # (time, lat, lon)

        # 確保 lat/lon 一致（保險，可視情況註解）
        if da.sizes["lat"] != nlat or da.sizes["lon"] != nlon:
            raise ValueError(f"Lat/Lon size mismatch in file: {f}")

        # 資料轉 float32，省記憶體
        data_list.append(da.values.astype(np.float32))

        if "time" not in ds:
            raise KeyError(f"'time' coordinate not found in file: {f}")

        # decode_cf 確保時間是真正 datetime
        t = xr.decode_cf(ds)["time"].values
        time_list.append(t.astype("datetime64[ns]"))

# 5. 串起來 → (ntot, lat, lon) & DatetimeIndex
combined = np.concatenate(data_list, axis=0)   # (ntot, nlat, nlon)
time_array = np.concatenate(time_list, axis=0) # (ntot,)

if combined.shape[0] != time_array.shape[0]:
    raise ValueError(
        f"time length ({time_array.shape[0]}) "
        f"!= data length ({combined.shape[0]})"
    )

# 依時間排序（通常已排序，這裡是保險）
sort_idx = np.argsort(time_array)
combined = combined[sort_idx]
time_array = time_array[sort_idx]

time_index = pd.to_datetime(time_array)

print(f"Combined data shape: {combined.shape}")
print(f"Time index: {time_index[0]} -> {time_index[-1]} (len={len(time_index)})")

# 6. 攤平成 cell × time
ntot, nlat, nlon = combined.shape
ncell = nlat * nlon

# (cell, time)
y_all = combined.reshape(ntot, ncell).T

# 建立每個 cell 的 (lon, lat)
lon_grid, lat_grid = np.meshgrid(lon, lat)
gg = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])  # (cell, 2)

print(f"y_all shape: {y_all.shape}  (cells x time)")
print(f"gg shape: {gg.shape}        (cells x [lon, lat])")


Data folder found: nc4
Found 548 files.
First 5 files: ['nc4/1980-01.nc4', 'nc4/1980-02.nc4', 'nc4/1980-03.nc4', 'nc4/1980-04.nc4', 'nc4/1980-05.nc4']
Data variables in first file:
['T2M', 'T2MDEW', 'Var_T2M', 'T2MWET']
Coordinates:
{'lon': (576,), 'time': (1,), 'lat': (361,)}
Confirmed var_name = T2M
Lat: 361, Lon: 576


Combining: 100%|██████████| 548/548 [00:10<00:00, 54.19it/s]


Combined data shape: (548, 361, 576)
Time index: 1980-01-01 00:30:00 -> 2025-08-01 00:30:00 (len=548)
y_all shape: (207936, 548)  (cells x time)
gg shape: (207936, 2)        (cells x [lon, lat])


# 限制範圍抽樣 in USA

## 100個格點

In [3]:
# ===== 只看美國本土範圍，並從中抽樣 100 個格點 =====
import numpy as np

# gg: (ncell, 2)；第 0 欄是 lon，第 1 欄是 lat
lon_all = gg[:, 0].astype(float)
lat_all = gg[:, 1].astype(float)

# 如果經度是 0~360，轉成 -180~180
lon_all_180 = ((lon_all + 180) % 360) - 180

# 美國本土 48 州的大致範圍
lon_min, lon_max = -125, -66
lat_min, lat_max = 24, 50

# 建立遮罩：只保留在美國本土範圍內的格點
mask_us_mainland = (
    (lon_all_180 >= lon_min) & (lon_all_180 <= lon_max) &
    (lat_all      >= lat_min) & (lat_all      <= lat_max)
)

idx_us_mainland = np.where(mask_us_mainland)[0]
print(f"美國本土範圍內的格點數量: {len(idx_us_mainland)}")

if len(idx_us_mainland) == 0:
    raise ValueError("在設定的美國本土範圍內沒有格點，請調整 lon_min/max 或 lat_min/max。")

# 只保留美國本土子集合
y_us = y_all[idx_us_mainland, :]  # (n_us, ntot)
coords_us = np.column_stack([
    lon_all_180[idx_us_mainland],
    lat_all[idx_us_mainland]
])

print(f"美國本土子集合 y_us shape: {y_us.shape}")
print(f"美國本土子集合 coords_us shape: {coords_us.shape}")

# ===== 抽樣 100 個格點 =====
N_SAMPLE_TARGET = 100
n_sample = min(N_SAMPLE_TARGET, len(idx_us_mainland))

np.random.seed(42)  # 為了可重現
sample_idx_in_us = np.random.choice(
    len(idx_us_mainland),
    size=n_sample,
    replace=False
)

# 原始 global index
sample_idx_global = idx_us_mainland[sample_idx_in_us]

# 抽樣後資料
y_sample = y_all[sample_idx_global, :]  # (n_sample, ntot)
coords_sample = np.column_stack([
    lon_all_180[sample_idx_global],
    lat_all[sample_idx_global]
])

lon_us_sample = coords_sample[:, 0]
lat_us_sample = coords_sample[:, 1]

print(f"抽樣 {n_sample} 個美國本土格點")
print("前 5 個 sample index (global):", sample_idx_global[:5])
print("前 5 個 sample 座標 (lon, lat):")
for i in range(min(5, n_sample)):
    print(f"  #{i}: lon={lon_us_sample[i]:.2f}, lat={lat_us_sample[i]:.2f}")

美國本土範圍內的格點數量: 5035
美國本土子集合 y_us shape: (5035, 548)
美國本土子集合 coords_us shape: (5035, 2)
抽樣 100 個美國本土格點
前 5 個 sample index (global): [153335 159677 145263 156825 138939]
前 5 個 sample 座標 (lon, lat):
  #0: lon=-105.62, lat=43.00
  #1: lon=-101.88, lat=48.50
  #2: lon=-110.62, lat=36.00
  #3: lon=-84.38, lat=46.00
  #4: lon=-103.12, lat=30.50


### DLinear

#### tune par (grid search)

In [4]:
import json
import time
import gc
import os
import logging
import contextlib
from pathlib import Path

import numpy as np
import pandas as pd
import optuna

from darts import TimeSeries
from darts.models import DLinearModel
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


print("\n===== DLinear hyperparameter search with Optuna (train/val/test = 70/15/15) =====")

optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

# Global settings & search controls
RUN_SEARCH = False  # False 的話會嘗試讀取 BEST_RESULT_PATH，讀到就跳過 search
N_TRIALS = 500
TIMEOUT_SECONDS = None

USE_COVARIATES = False
USE_REVIN = True
RANDOM_STATE = 42

PL_TRAINER_KWARGS = {
    "enable_progress_bar": False,
    "logger": False,
    "enable_checkpointing": False,
    "enable_model_summary": False,
}

SAVE_DIR = Path(".")
BEST_RESULT_PATH = SAVE_DIR / "best_dlinear_100sites_result.json"
BEST_PARAMS_PATH = SAVE_DIR / "best_dlinear_100sites_params.json"


def _json_dump(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def _json_load(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _cleanup_torch_cache() -> None:
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def _ts_to_2d(ts: TimeSeries) -> np.ndarray:
    arr = np.asarray(ts.all_values(copy=False))
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr


def _build_model_kwargs(
    in_len: int,
    out_len: int,
    ksize: int,
    n_epochs: int,
    bs: int,
    lr: float,
    wd: float,
    const_init: bool,
) -> dict:
    return {
        "input_chunk_length": int(in_len),
        "output_chunk_length": int(out_len),
        "kernel_size": int(ksize),
        "n_epochs": int(n_epochs),
        "random_state": int(RANDOM_STATE),
        "use_reversible_instance_norm": bool(USE_REVIN),
        "batch_size": int(bs),
        "optimizer_kwargs": {
            "lr": float(lr),
            "weight_decay": float(wd),
        },
        "const_init": bool(const_init),
        "pl_trainer_kwargs": PL_TRAINER_KWARGS,
        "log_tensorboard": False,
        "save_checkpoints": False,
    }


def _compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    r2 = r2_score(y_true, y_pred)
    return {
        "rmse": float(rmse),
        "mse": float(mse),
        "mae": float(mae),
        "r2": float(r2),
    }


def _silent_call(func, *args, **kwargs):
    with open(os.devnull, "w") as fnull:
        with contextlib.redirect_stdout(fnull), contextlib.redirect_stderr(fnull):
            return func(*args, **kwargs)


SAMPLE_SIZE = int(n_sample)
T = int(y_sample.shape[1])

ts_df = pd.DataFrame(
    y_sample.T,
    index=time_index,
    columns=[f"cell_{i}" for i in range(SAMPLE_SIZE)]
).astype("float32")

print("Total time steps:", T)
print("ts_df shape:", ts_df.shape)

month_values = time_index.month.astype("float32")
month_df = pd.DataFrame({"month": month_values}, index=time_index)
month_ts = TimeSeries.from_dataframe(month_df.astype("float32"))

train_frac = 0.7
val_frac = 0.15
test_frac = 0.15

cut_train = int(T * train_frac)
cut_val = int(T * (train_frac + val_frac))

idx = ts_df.index
split_time_train = idx[cut_train]
split_time_val = idx[cut_val]

ts_all_raw = TimeSeries.from_dataframe(ts_df)
trainval_raw, test_raw = ts_all_raw.split_before(split_time_val)
train_raw, val_raw = trainval_raw.split_before(split_time_train)

print(f"Train len (raw): {len(train_raw)}, Val len: {len(val_raw)}, Test len: {len(test_raw)}")

month_trainval, month_test = month_ts.split_before(split_time_val)
month_train, month_val = month_trainval.split_before(split_time_train)

assert len(month_train) == len(train_raw)
assert len(month_val) == len(val_raw)
assert len(month_test) == len(test_raw)

train_df = ts_df.iloc[:cut_train]
mean_vec = train_df.mean(axis=0)
std_vec = train_df.std(axis=0).replace(0.0, 1.0)

ts_df_scaled = (ts_df - mean_vec) / std_vec
ts_all_scaled = TimeSeries.from_dataframe(ts_df_scaled)

trainval_ts, test_ts = ts_all_scaled.split_before(split_time_val)
train_ts, val_ts = trainval_ts.split_before(split_time_train)

T_train, T_val, T_test = len(train_ts), len(val_ts), len(test_ts)
print("\n[Scaled lengths] T_train =", T_train, "T_val =", T_val, "T_test =", T_test)

val_true_raw = ts_df.to_numpy(dtype=np.float32)[cut_train:cut_val, :]
test_true_raw = ts_df.to_numpy(dtype=np.float32)[cut_val:, :]


def inverse_scale(x: np.ndarray) -> np.ndarray:
    return x * std_vec.values + mean_vec.values


SEARCH_SPACE = {
    "INPUT_CHUNK_LENGTH": [24, 36, 48],
    "OUTPUT_CHUNK_LENGTH": [12, 24],
    "KERNEL_SIZE": [15, 25, 35],
    "N_EPOCHS": [30, 50],
    "BATCH_SIZE": [32, 64, 128],
    "LR": [1e-4, 2e-4, 3e-4],
    "WEIGHT_DECAY": [0.0, 1e-5],
    "CONST_INIT": [True, False],
}

keys = list(SEARCH_SPACE.keys())

if (not RUN_SEARCH) and BEST_RESULT_PATH.exists():
    best_loaded = _json_load(BEST_RESULT_PATH)

    print("\n===== LOADED BEST RESULT (skip search) =====")
    print("Best VAL metrics:")
    print(f"  RMSE = {best_loaded['rmse_val']:.6f}")
    print(f"  MSE  = {best_loaded['mse_val']:.6f}")
    print(f"  MAE  = {best_loaded['mae_val']:.6f}")
    print(f"  R2   = {best_loaded['r2_val']:.6f}")

    print("\nBest TEST metrics:")
    print(f"  RMSE = {best_loaded['rmse_test']:.6f}")
    print(f"  MSE  = {best_loaded['mse_test']:.6f}")
    print(f"  MAE  = {best_loaded['mae_test']:.6f}")
    print(f"  R2   = {best_loaded['r2_test']:.6f}")

    print("\nBest params:")
    for k in keys:
        print(f"  {k} = {best_loaded['params'][k]}")
    print(f"  USE_REVIN = {best_loaded['params']['USE_REVIN']}")
    print(f"  USE_COVARIATES = {best_loaded['params']['USE_COVARIATES']}")
    print(f"  RANDOM_STATE = {best_loaded['params']['RANDOM_STATE']}")
else:
    def objective(trial: optuna.Trial) -> float:
        print(f"[Trial {trial.number + 1}/{N_TRIALS}]", flush=True)

        in_len = trial.suggest_categorical(
            "INPUT_CHUNK_LENGTH",
            SEARCH_SPACE["INPUT_CHUNK_LENGTH"]
        )
        out_len = trial.suggest_categorical(
            "OUTPUT_CHUNK_LENGTH",
            SEARCH_SPACE["OUTPUT_CHUNK_LENGTH"]
        )
        ksize = trial.suggest_categorical(
            "KERNEL_SIZE",
            SEARCH_SPACE["KERNEL_SIZE"]
        )
        n_epochs = trial.suggest_categorical(
            "N_EPOCHS",
            SEARCH_SPACE["N_EPOCHS"]
        )
        bs = trial.suggest_categorical(
            "BATCH_SIZE",
            SEARCH_SPACE["BATCH_SIZE"]
        )
        lr = trial.suggest_categorical(
            "LR",
            SEARCH_SPACE["LR"]
        )
        wd = trial.suggest_categorical(
            "WEIGHT_DECAY",
            SEARCH_SPACE["WEIGHT_DECAY"]
        )
        const_init = trial.suggest_categorical(
            "CONST_INIT",
            SEARCH_SPACE["CONST_INIT"]
        )

        model_kwargs = _build_model_kwargs(
            in_len=in_len,
            out_len=out_len,
            ksize=ksize,
            n_epochs=n_epochs,
            bs=bs,
            lr=lr,
            wd=wd,
            const_init=const_init,
        )

        model = DLinearModel(**model_kwargs)

        fit_kwargs = {
            "series": train_ts,
            "verbose": False,
        }
        pred_val_kwargs = {
            "n": T_val,
            "verbose": False,
            "show_warnings": False,
        }

        if USE_COVARIATES:
            fit_kwargs["past_covariates"] = month_train
            fit_kwargs["future_covariates"] = month_train
            pred_val_kwargs["past_covariates"] = month_trainval
            pred_val_kwargs["future_covariates"] = month_trainval

        try:
            _silent_call(model.fit, **fit_kwargs)
            pred_val = _silent_call(model.predict, **pred_val_kwargs)
        except Exception:
            del model
            gc.collect()
            _cleanup_torch_cache()
            raise optuna.TrialPruned()

        pred_val_scaled = _ts_to_2d(pred_val)
        pred_val_raw = inverse_scale(pred_val_scaled)

        y_val_true = val_true_raw.reshape(-1, SAMPLE_SIZE)
        y_val_pred = pred_val_raw.reshape(-1, SAMPLE_SIZE)

        val_metrics = _compute_metrics(y_val_true, y_val_pred)

        trial.set_user_attr("rmse_val", val_metrics["rmse"])
        trial.set_user_attr("mse_val", val_metrics["mse"])
        trial.set_user_attr("mae_val", val_metrics["mae"])
        trial.set_user_attr("r2_val", val_metrics["r2"])
        trial.set_user_attr("USE_REVIN", bool(USE_REVIN))
        trial.set_user_attr("USE_COVARIATES", bool(USE_COVARIATES))
        trial.set_user_attr("RANDOM_STATE", int(RANDOM_STATE))

        del model
        del pred_val, pred_val_scaled, pred_val_raw
        del y_val_true, y_val_pred, val_metrics
        gc.collect()
        _cleanup_torch_cache()

        return trial.user_attrs["rmse_val"]

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )

    study.optimize(
        objective,
        n_trials=N_TRIALS,
        timeout=TIMEOUT_SECONDS,
        gc_after_trial=True,
        show_progress_bar=False,
    )

    print("\n===== BEST PARAMS (by VAL RMSE) =====")

    successful_trials = [
        t for t in study.trials
        if t.state == optuna.trial.TrialState.COMPLETE
    ]

    if len(successful_trials) == 0:
        print("No successful run in the search space.")
    else:
        best_trial = study.best_trial

        best_params = {
            "INPUT_CHUNK_LENGTH": best_trial.params["INPUT_CHUNK_LENGTH"],
            "OUTPUT_CHUNK_LENGTH": best_trial.params["OUTPUT_CHUNK_LENGTH"],
            "KERNEL_SIZE": best_trial.params["KERNEL_SIZE"],
            "N_EPOCHS": best_trial.params["N_EPOCHS"],
            "BATCH_SIZE": best_trial.params["BATCH_SIZE"],
            "LR": float(best_trial.params["LR"]),
            "WEIGHT_DECAY": float(best_trial.params["WEIGHT_DECAY"]),
            "CONST_INIT": bool(best_trial.params["CONST_INIT"]),
            "USE_REVIN": bool(USE_REVIN),
            "USE_COVARIATES": bool(USE_COVARIATES),
            "RANDOM_STATE": int(RANDOM_STATE),
        }

        print("Best VAL metrics:")
        print(f"  RMSE = {best_trial.user_attrs['rmse_val']:.6f}")
        print(f"  MSE  = {best_trial.user_attrs['mse_val']:.6f}")
        print(f"  MAE  = {best_trial.user_attrs['mae_val']:.6f}")
        print(f"  R2   = {best_trial.user_attrs['r2_val']:.6f}")

        best_model_kwargs = _build_model_kwargs(
            in_len=best_params["INPUT_CHUNK_LENGTH"],
            out_len=best_params["OUTPUT_CHUNK_LENGTH"],
            ksize=best_params["KERNEL_SIZE"],
            n_epochs=best_params["N_EPOCHS"],
            bs=best_params["BATCH_SIZE"],
            lr=best_params["LR"],
            wd=best_params["WEIGHT_DECAY"],
            const_init=best_params["CONST_INIT"],
        )

        best_model = DLinearModel(**best_model_kwargs)

        fit_best_kwargs = {
            "series": trainval_ts,
            "verbose": False,
        }
        pred_test_kwargs = {
            "n": T_test,
            "verbose": False,
            "show_warnings": False,
        }

        if USE_COVARIATES:
            fit_best_kwargs["past_covariates"] = month_trainval
            fit_best_kwargs["future_covariates"] = month_trainval
            pred_test_kwargs["past_covariates"] = month_ts
            pred_test_kwargs["future_covariates"] = month_ts

        _silent_call(best_model.fit, **fit_best_kwargs)
        pred_test = _silent_call(best_model.predict, **pred_test_kwargs)

        pred_test_scaled = _ts_to_2d(pred_test)
        pred_test_raw = inverse_scale(pred_test_scaled)

        y_test_true = test_true_raw.reshape(-1, SAMPLE_SIZE)
        y_test_pred = pred_test_raw.reshape(-1, SAMPLE_SIZE)

        test_metrics = _compute_metrics(y_test_true, y_test_pred)

        best_result_payload = {
            "rmse_val": float(best_trial.user_attrs["rmse_val"]),
            "mse_val": float(best_trial.user_attrs["mse_val"]),
            "mae_val": float(best_trial.user_attrs["mae_val"]),
            "r2_val": float(best_trial.user_attrs["r2_val"]),
            "rmse_test": float(test_metrics["rmse"]),
            "mse_test": float(test_metrics["mse"]),
            "mae_test": float(test_metrics["mae"]),
            "r2_test": float(test_metrics["r2"]),
            "params": best_params,
            "search_space": SEARCH_SPACE,
            "n_trials": int(N_TRIALS),
            "best_trial_number": int(best_trial.number),
            "split_ratio": {
                "train": train_frac,
                "val": val_frac,
                "test": test_frac,
            },
        }

        print("\nBest TEST metrics:")
        print(f"  RMSE = {best_result_payload['rmse_test']:.6f}")
        print(f"  MSE  = {best_result_payload['mse_test']:.6f}")
        print(f"  MAE  = {best_result_payload['mae_test']:.6f}")
        print(f"  R2   = {best_result_payload['r2_test']:.6f}")

        print("\nBest params:")
        for k in keys:
            print(f"  {k} = {best_params[k]}")
        print(f"  USE_REVIN = {best_params['USE_REVIN']}")
        print(f"  USE_COVARIATES = {best_params['USE_COVARIATES']}")
        print(f"  RANDOM_STATE = {best_params['RANDOM_STATE']}")
        print(f"  BEST_TRIAL_NUMBER = {best_trial.number}")

        _json_dump(best_result_payload, BEST_RESULT_PATH)
        _json_dump(best_params, BEST_PARAMS_PATH)

        print("\nSaved:")
        print(f"  - {BEST_RESULT_PATH.resolve()}")
        print(f"  - {BEST_PARAMS_PATH.resolve()}")

        del best_model
        del pred_test, pred_test_scaled, pred_test_raw
        del y_test_true, y_test_pred, test_metrics
        gc.collect()
        _cleanup_torch_cache()

/home/jundian/installers/yes/envs/geospatial-neural-adapter/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



===== DLinear hyperparameter search with Optuna (train/val/test = 70/15/15) =====
Total time steps: 548
ts_df shape: (548, 100)
Train len (raw): 383, Val len: 82, Test len: 83

[Scaled lengths] T_train = 383 T_val = 82 T_test = 83

===== LOADED BEST RESULT (skip search) =====
Best VAL metrics:
  RMSE = 1.888609
  MSE  = 3.566845
  MAE  = 1.414827
  R2   = 0.856411

Best TEST metrics:
  RMSE = 1.700174
  MSE  = 2.890592
  MAE  = 1.256546
  R2   = 0.911410

Best params:
  INPUT_CHUNK_LENGTH = 48
  OUTPUT_CHUNK_LENGTH = 24
  KERNEL_SIZE = 35
  N_EPOCHS = 50
  BATCH_SIZE = 128
  LR = 0.0001
  WEIGHT_DECAY = 0.0
  CONST_INIT = True
  USE_REVIN = True
  USE_COVARIATES = False
  RANDOM_STATE = 42


#### formal

In [5]:
# ============================================================
# 8) Re-run DLinear with BEST params (train+val -> test)
#    Priority:
#      (1) load from best_dlinear_100sites_params.json
#      (2) fallback: load params from best_dlinear_100sites_result.json
# ============================================================
from pathlib import Path
import json
import gc
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from darts.models import DLinearModel

BEST_PARAMS_PATH = Path("best_dlinear_100sites_params.json")
BEST_RESULT_PATH = Path("best_dlinear_100sites_result.json")
RERUN_METRICS_PATH = Path("best_dlinear_100sites_rerun_metrics.json")


def _json_load(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _json_dump(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def _cleanup_torch_cache() -> None:
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


bp = None

if BEST_PARAMS_PATH.exists():
    bp = _json_load(BEST_PARAMS_PATH)
elif BEST_RESULT_PATH.exists():
    best_result_loaded = _json_load(BEST_RESULT_PATH)
    bp = best_result_loaded.get("params", None)

if bp is None:
    raise RuntimeError(
        "No best params found. Please make sure best_dlinear_100sites_params.json "
        "or best_dlinear_100sites_result.json exists."
    )

print("\n===== RE-RUN DLinear with BEST params (train+val/test = 85/15 under 70/15/15 split) =====")
print("Best params used:")
for k in keys:
    if k in bp:
        print(f"  {k} = {bp[k]}")
print(f"  USE_REVIN = {bp.get('USE_REVIN', USE_REVIN)}")
print(f"  USE_COVARIATES = {bp.get('USE_COVARIATES', USE_COVARIATES)}")
print(f"  RANDOM_STATE = {bp.get('RANDOM_STATE', RANDOM_STATE)}")

model_best_kwargs = dict(
    input_chunk_length=int(bp["INPUT_CHUNK_LENGTH"]),
    output_chunk_length=int(bp["OUTPUT_CHUNK_LENGTH"]),
    kernel_size=int(bp["KERNEL_SIZE"]),
    n_epochs=int(bp["N_EPOCHS"]),
    random_state=int(bp.get("RANDOM_STATE", RANDOM_STATE)),
    use_reversible_instance_norm=bool(bp.get("USE_REVIN", USE_REVIN)),
    batch_size=int(bp["BATCH_SIZE"]),
    optimizer_kwargs={
        "lr": float(bp["LR"]),
        "weight_decay": float(bp["WEIGHT_DECAY"]),
    },
    const_init=bool(bp["CONST_INIT"]),
    pl_trainer_kwargs=PL_TRAINER_KWARGS,
    log_tensorboard=False,
    save_checkpoints=False,
)

model_best = DLinearModel(**model_best_kwargs)

fit_kwargs_best = {
    "series": trainval_ts,
    "verbose": False,
}
pred_kwargs_best = {
    "n": T_test,
    "verbose": False,
    "show_warnings": False,
}

if bool(bp.get("USE_COVARIATES", USE_COVARIATES)):
    fit_kwargs_best["past_covariates"] = month_trainval
    fit_kwargs_best["future_covariates"] = month_trainval
    pred_kwargs_best["past_covariates"] = month_ts
    pred_kwargs_best["future_covariates"] = month_ts

model_best.fit(**fit_kwargs_best)
pred_best = model_best.predict(**pred_kwargs_best)

pred_best_scaled = _ts_to_2d(pred_best)
pred_best_raw = inverse_scale(pred_best_scaled)

y_test_true_best = test_true_raw.reshape(-1, SAMPLE_SIZE)
y_test_pred_best = pred_best_raw.reshape(-1, SAMPLE_SIZE)

mse_best = mean_squared_error(y_test_true_best, y_test_pred_best)
mae_best = mean_absolute_error(y_test_true_best, y_test_pred_best)
rmse_best = float(np.sqrt(mse_best))
r2_best = r2_score(y_test_true_best, y_test_pred_best)

metrics_best = pd.DataFrame({
    "Model": ["DLINEAR(best)"],
    "Split": ["TEST"],
    "RMSE": [rmse_best],
    "MSE": [float(mse_best)],
    "MAE": [float(mae_best)],
    "R2": [float(r2_best)],
})

print("\n===== BEST DLINEAR RESULTS (re-run) =====")
print(metrics_best.to_string(index=False))

rerun_payload = {
    "Model": "DLINEAR(best)",
    "Split": "TEST",
    "RMSE": float(rmse_best),
    "MSE": float(mse_best),
    "MAE": float(mae_best),
    "R2": float(r2_best),
    "params_used": bp,
    "train_series_used": "trainval_ts",
    "test_series_used": "test_ts",
    "split_ratio": {
        "train": 0.7,
        "val": 0.15,
        "test": 0.15,
    },
}

_json_dump(rerun_payload, RERUN_METRICS_PATH)
print(f"\nSaved re-run metrics: {RERUN_METRICS_PATH.resolve()}")

del model_best, pred_best, pred_best_scaled, pred_best_raw
del y_test_true_best, y_test_pred_best, mse_best, mae_best, rmse_best, r2_best
gc.collect()
_cleanup_torch_cache()


===== RE-RUN DLinear with BEST params (train+val/test = 85/15 under 70/15/15 split) =====
Best params used:
  INPUT_CHUNK_LENGTH = 48
  OUTPUT_CHUNK_LENGTH = 24
  KERNEL_SIZE = 35
  N_EPOCHS = 50
  BATCH_SIZE = 128
  LR = 0.0001
  WEIGHT_DECAY = 0.0
  CONST_INIT = True
  USE_REVIN = True
  USE_COVARIATES = False
  RANDOM_STATE = 42

===== BEST DLINEAR RESULTS (re-run) =====
        Model Split     RMSE      MSE      MAE      R2
DLINEAR(best)  TEST 1.700174 2.890592 1.256546 0.91141

Saved re-run metrics: /home/jundian/Research-Project/project/best_dlinear_100sites_rerun_metrics.json


## Plot Residual Covariate

In [6]:
# # Residual covariance with reference site
# import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.tri as mtri

# def pick_center_site(coords):
#     coords = np.asarray(coords, dtype=np.float64)
#     center = np.array([np.mean(coords[:, 0]), np.mean(coords[:, 1])], dtype=np.float64)
#     d2 = np.sum((coords - center) ** 2, axis=1)
#     ref_idx = int(np.argmin(d2))
#     return ref_idx

# def cov_with_reference(R, ref_idx, ddof=1):
#     R = np.asarray(R, dtype=np.float64)
#     R_tc = R - np.mean(R, axis=0, keepdims=True)
#     ref = R_tc[:, ref_idx]
#     cov_vec = (R_tc * ref[:, None]).sum(axis=0) / (R_tc.shape[0] - ddof)
#     return cov_vec

# coords = coords_sample.astype(np.float64)
# lat = coords[:, 1]

# ref_idx = pick_center_site(coords)

# covref_dlinear = cov_with_reference(R_dlinear, ref_idx)
# covref_combo = cov_with_reference(R_combo, ref_idx)
# covref_diff = covref_combo - covref_dlinear

# triang = mtri.Triangulation(lon, lat)

# fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# tcf0 = axes[0].tricontourf(triang, covref_dlinear, levels=20)
# axes[0].scatter(lon[ref_idx], lat[ref_idx], marker="x", s=80)
# axes[0].set_title("Ref-site Covariance: DLinear(best)")
# axes[0].set_xlabel("Longitude")
# axes[0].set_ylabel("Latitude")
# plt.colorbar(tcf0, ax=axes[0], fraction=0.046, pad=0.04)

# tcf1 = axes[1].tricontourf(triang, covref_combo, levels=20)
# axes[1].scatter(lon[ref_idx], lat[ref_idx], marker="x", s=80)
# axes[1].set_title("Ref-site Covariance: DLinear + autoFRK")
# axes[1].set_xlabel("Longitude")
# axes[1].set_ylabel("Latitude")
# plt.colorbar(tcf1, ax=axes[1], fraction=0.046, pad=0.04)

# tcf2 = axes[2].tricontourf(triang, covref_diff, levels=20)
# axes[2].scatter(lon[ref_idx], lat[ref_idx], marker="x", s=80)
# axes[2].set_title("Ref-site Covariance Difference")
# axes[2].set_xlabel("Longitude")
# axes[2].set_ylabel("Latitude")
# plt.colorbar(tcf2, ax=axes[2], fraction=0.046, pad=0.04)

# plt.tight_layout()
# plt.show()

# print("Reference site index:", ref_idx)
# print("Reference site coord:", coords[ref_idx])

## autoFRK (100)

In [7]:
# ============================================================
# 9) DLinear + autoFRK rolling residual forecast (100 sites)
# ============================================================
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from autoFRK import AutoFRK
from darts.models import DLinearModel
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("\n===== DLinear + autoFRK rolling residual forecast (100 sites) =====")

BEST_PARAMS_PATH = Path("best_dlinear_100sites_params.json")
BEST_RESULT_PATH = Path("best_dlinear_100sites_result.json")
FRK_METRICS_PATH = Path("dlinear_100sites_autofrk_rolling_test_metrics.json")


def _json_load(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _json_dump(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def _cleanup_torch_cache() -> None:
    try:
        import torch as _torch
        if _torch.cuda.is_available():
            _torch.cuda.empty_cache()
    except Exception:
        pass


def _ts_to_2d(ts):
    arr = np.asarray(ts.all_values(copy=False))
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr.astype(np.float64, copy=False)


def inverse_scale_2d(x_TN):
    return x_TN * std_arr[None, :] + mean_arr[None, :]


# ------------------------------------------------------------
# 0) Load best params
# ------------------------------------------------------------
bp = None
if BEST_PARAMS_PATH.exists():
    bp = _json_load(BEST_PARAMS_PATH)
elif BEST_RESULT_PATH.exists():
    bp = _json_load(BEST_RESULT_PATH).get("params", None)

if bp is None:
    raise RuntimeError(
        "No best params found. Please make sure best_dlinear_100sites_params.json "
        "or best_dlinear_100sites_result.json exists."
    )

# ------------------------------------------------------------
# 1) Rebuild / reuse best DLinear model on train+val -> test
# ------------------------------------------------------------
need_refit_best_model = (
    ("model_best" not in globals()) or
    ("pred_best_raw" not in globals())
)

if need_refit_best_model:
    print("model_best / pred_best_raw not found in memory. Re-fitting best DLinear on train+val...")

    model_best_kwargs = dict(
        input_chunk_length=int(bp["INPUT_CHUNK_LENGTH"]),
        output_chunk_length=int(bp["OUTPUT_CHUNK_LENGTH"]),
        kernel_size=int(bp["KERNEL_SIZE"]),
        n_epochs=int(bp["N_EPOCHS"]),
        random_state=int(bp.get("RANDOM_STATE", RANDOM_STATE)),
        use_reversible_instance_norm=bool(bp.get("USE_REVIN", USE_REVIN)),
        batch_size=int(bp["BATCH_SIZE"]),
        optimizer_kwargs={
            "lr": float(bp["LR"]),
            "weight_decay": float(bp["WEIGHT_DECAY"]),
        },
        const_init=bool(bp["CONST_INIT"]),
        pl_trainer_kwargs=PL_TRAINER_KWARGS,
        log_tensorboard=False,
        save_checkpoints=False,
    )

    model_best = DLinearModel(**model_best_kwargs)

    fit_kwargs_best = {
        "series": trainval_ts,
        "verbose": False,
    }
    pred_kwargs_best = {
        "n": T_test,
        "verbose": False,
        "show_warnings": False,
    }

    if bool(bp.get("USE_COVARIATES", USE_COVARIATES)):
        fit_kwargs_best["past_covariates"] = month_trainval
        fit_kwargs_best["future_covariates"] = month_trainval
        pred_kwargs_best["past_covariates"] = month_ts
        pred_kwargs_best["future_covariates"] = month_ts

    model_best.fit(**fit_kwargs_best)
    pred_best = model_best.predict(**pred_kwargs_best)

    pred_best_scaled = _ts_to_2d(pred_best)
    pred_best_raw = inverse_scale(pred_best_scaled)

    del pred_best, pred_best_scaled
    gc.collect()
    _cleanup_torch_cache()

# ------------------------------------------------------------
# 2) Basic objects
# ------------------------------------------------------------
coords = coords_sample.astype(np.float64)
loc = torch.from_numpy(coords).to(dtype=torch.float64, device="cpu")

N = coords.shape[0]
H = T_test

if N != 100:
    raise ValueError(f"這一段是 100 個格點 FRK 版本，目前 N = {N}，不是 100。")

mean_arr = mean_vec.values.astype(np.float64)
std_arr = std_vec.values.astype(np.float64)

if mean_arr.shape[0] != N:
    raise ValueError(f"mean_vec 長度 {mean_arr.shape[0]} 與格點數 N={N} 不一致")
if std_arr.shape[0] != N:
    raise ValueError(f"std_vec 長度 {std_arr.shape[0]} 與格點數 N={N} 不一致")
if ts_df.shape[1] != N:
    raise ValueError(f"ts_df 欄數 {ts_df.shape[1]} 與格點數 N={N} 不一致")

input_chunk_length_best = int(bp["INPUT_CHUNK_LENGTH"])

# ------------------------------------------------------------
# 3) Build residual history from train+val period
# ------------------------------------------------------------
y_hist_raw_TN = ts_df.iloc[:cut_val, :].to_numpy(dtype=np.float64)
hist_index = ts_df.index[:cut_val]
T_hist = y_hist_raw_TN.shape[0]

hist_forecast_kwargs = {
    "series": trainval_ts,
    "start": input_chunk_length_best,
    "forecast_horizon": 1,
    "stride": 1,
    "retrain": False,
    "last_points_only": True,
    "verbose": False,
}

if bool(bp.get("USE_COVARIATES", USE_COVARIATES)):
    hist_forecast_kwargs["past_covariates"] = month_trainval
    hist_forecast_kwargs["future_covariates"] = month_trainval

pred_hist = model_best.historical_forecasts(**hist_forecast_kwargs)

if isinstance(pred_hist, list):
    if len(pred_hist) == 0:
        raise RuntimeError("historical_forecasts 回傳空 list，無法建立 residual history。")
    pred_hist_ts = pred_hist[0]
    for k in range(1, len(pred_hist)):
        pred_hist_ts = pred_hist_ts.append(pred_hist[k])
else:
    pred_hist_ts = pred_hist

pred_hist_scaled_TN = _ts_to_2d(pred_hist_ts)
pred_time_index = pred_hist_ts.time_index

trend_hat_hist_raw_TN = np.full((T_hist, N), np.nan, dtype=np.float64)

pos = hist_index.get_indexer(pred_time_index)
valid = pos >= 0
pred_hist_scaled_TN = pred_hist_scaled_TN[valid]
pos = pos[valid]

pred_hist_raw_TN = inverse_scale_2d(pred_hist_scaled_TN)
trend_hat_hist_raw_TN[pos] = pred_hist_raw_TN

R_hist_raw_TN = y_hist_raw_TN - trend_hat_hist_raw_TN
R_hist_raw_NT = R_hist_raw_TN.T

nan_ratio = float(np.isnan(R_hist_raw_NT).mean())
print("Residual history shape:", R_hist_raw_NT.shape, "| NaN ratio:", nan_ratio)

valid_cols = ~np.isnan(R_hist_raw_NT).any(axis=0)
R_hist_valid_NT = R_hist_raw_NT[:, valid_cols]

if R_hist_valid_NT.shape[1] < 2:
    raise RuntimeError("有效 residual 歷史長度不足，無法做 rolling latent forecast。")

# ------------------------------------------------------------
# 4) DLinear test trend
# ------------------------------------------------------------
trend_test_raw = pred_best_raw.astype(np.float64)
if trend_test_raw.shape != (H, N):
    raise ValueError(f"pred_best_raw shape {trend_test_raw.shape} 應為 {(H, N)}")

y_test_true_combo = test_true_raw.reshape(H, N).astype(np.float64)

# ------------------------------------------------------------
# 5) Rolling residual forecast with autoFRK
# ------------------------------------------------------------
afrk = AutoFRK(dtype=torch.float64, device="cpu")

Rhat_roll_NH = np.zeros((N, H), dtype=np.float64)
K_list = []

R_hist_roll_NT = R_hist_valid_NT.copy()

for h in range(H):
    data_hist = torch.from_numpy(R_hist_roll_NT).to(dtype=torch.float64)

    result_h = afrk.forward(
        data=data_hist,
        loc=loc,
        method="EM",
        maxit=50,
        tolerance=1e-6,
        n_neighbor=3,
        tps_method="spherical_fast",
    )

    w_hist = result_h["w"]
    w_hist_np = w_hist.detach().cpu().numpy()
    K, T_latent = w_hist_np.shape
    K_list.append(int(K))

    if T_latent < 2:
        raise RuntimeError(f"第 {h+1} 步 latent 歷史長度不足，無法建立 VAR(1)。")

    W0 = w_hist_np[:, :-1]
    W1 = w_hist_np[:, 1:]
    A = (W1 @ W0.T) @ np.linalg.pinv(W0 @ W0.T)

    w_last = w_hist_np[:, -1]
    w_next = (A @ w_last).reshape(K, 1)

    obj_next = dict(result_h)
    obj_next["w"] = torch.from_numpy(w_next).to(dtype=torch.float64)

    pred_res_h = afrk.predict(
        obj=obj_next,
        newloc=loc,
        se_report=False,
        tps_method="spherical_fast",
    )

    rhat_h = pred_res_h["pred.value"].detach().cpu().numpy()

    if rhat_h.ndim == 2 and rhat_h.shape == (N, 1):
        rhat_h = rhat_h[:, 0]
    elif rhat_h.ndim == 1 and rhat_h.shape[0] == N:
        pass
    else:
        raise ValueError(f"第 {h+1} 步 residual prediction shape 異常: {rhat_h.shape}")

    Rhat_roll_NH[:, h] = rhat_h

    y_true_h = y_test_true_combo[h, :]
    trend_h = trend_test_raw[h, :]
    r_true_h = y_true_h - trend_h

    R_hist_roll_NT = np.column_stack([R_hist_roll_NT, r_true_h])

# ------------------------------------------------------------
# 6) Final combined forecast
# ------------------------------------------------------------
Yhat_test_combo = trend_test_raw + Rhat_roll_NH.T

mse_combo = mean_squared_error(y_test_true_combo, Yhat_test_combo)
mae_combo = mean_absolute_error(y_test_true_combo, Yhat_test_combo)
rmse_combo = float(np.sqrt(mse_combo))
r2_combo = r2_score(y_test_true_combo, Yhat_test_combo)

mse_dlinear = mean_squared_error(y_test_true_combo, trend_test_raw)
mae_dlinear = mean_absolute_error(y_test_true_combo, trend_test_raw)
rmse_dlinear = float(np.sqrt(mse_dlinear))
r2_dlinear = r2_score(y_test_true_combo, trend_test_raw)

metrics_compare = pd.DataFrame({
    "Model": ["DLinear(best)", "DLinear + autoFRK rolling"],
    "Split": ["TEST", "TEST"],
    "RMSE": [rmse_dlinear, rmse_combo],
    "MSE": [float(mse_dlinear), float(mse_combo)],
    "MAE": [float(mae_dlinear), float(mae_combo)],
    "R2": [float(r2_dlinear), float(r2_combo)],
})

print("\n===== DLinear vs DLinear + autoFRK rolling =====")
print(metrics_compare.to_string(index=False))

frk_payload = {
    "dlinear_best": {
        "RMSE": float(rmse_dlinear),
        "MSE": float(mse_dlinear),
        "MAE": float(mae_dlinear),
        "R2": float(r2_dlinear),
    },
    "dlinear_autofrk_rolling": {
        "RMSE": float(rmse_combo),
        "MSE": float(mse_combo),
        "MAE": float(mae_combo),
        "R2": float(r2_combo),
    },
    "N": int(N),
    "H": int(H),
    "K_first_step": int(K_list[0]) if len(K_list) > 0 else None,
    "K_last_step": int(K_list[-1]) if len(K_list) > 0 else None,
    "residual_history_nan_ratio": float(nan_ratio),
    "residual_history_length_used": int(R_hist_valid_NT.shape[1]),
    "params_used": bp,
    "split_ratio": {
        "train": 0.7,
        "val": 0.15,
        "test": 0.15,
    },
}

_json_dump(frk_payload, FRK_METRICS_PATH)
print(f"\nSaved rolling FRK metrics: {FRK_METRICS_PATH.resolve()}")


===== DLinear + autoFRK rolling residual forecast (100 sites) =====
model_best / pred_best_raw not found in memory. Re-fitting best DLinear on train+val...


2026-03-25 21:12:38 - autoFRK.utils.logger - INFO: Successfully using device "cpu".
2026-03-25 21:12:38 - autoFRK.utils.logger - INFO: Calculate TPS with spherical_fast.


Residual history shape: (100, 465) | NaN ratio: 0.1032258064516129


2026-03-25 21:12:38 - autoFRK.utils.logger - INFO: Successfully using device "cpu".
2026-03-25 21:12:38 - autoFRK.utils.logger - INFO: Calculate TPS with spherical_fast.
2026-03-25 21:12:39 - autoFRK.utils.logger - INFO: Successfully using device "cpu".
2026-03-25 21:12:39 - autoFRK.utils.logger - INFO: Calculate TPS with spherical_fast.
2026-03-25 21:12:39 - autoFRK.utils.logger - INFO: Successfully using device "cpu".
2026-03-25 21:12:39 - autoFRK.utils.logger - INFO: Calculate TPS with spherical_fast.
2026-03-25 21:12:39 - autoFRK.utils.logger - INFO: Successfully using device "cpu".
2026-03-25 21:12:39 - autoFRK.utils.logger - INFO: Calculate TPS with spherical_fast.
2026-03-25 21:12:40 - autoFRK.utils.logger - INFO: Successfully using device "cpu".
2026-03-25 21:12:40 - autoFRK.utils.logger - INFO: Calculate TPS with spherical_fast.
2026-03-25 21:12:39 - autoFRK.utils.logger - INFO: Successfully using device "cpu".
2026-03-25 21:12:39 - autoFRK.utils.logger - INFO: Calculate TPS w


===== DLinear vs DLinear + autoFRK rolling =====
                    Model Split     RMSE      MSE      MAE       R2
            DLinear(best)  TEST 1.700174 2.890591 1.256546 0.911410
DLinear + autoFRK rolling  TEST 1.674329 2.803378 1.203352 0.936779

Saved rolling FRK metrics: /home/jundian/Research-Project/project/dlinear_100sites_autofrk_rolling_test_metrics.json


### FRK prediction 圖

In [8]:
# # ============================================================
# # 10) Plot FRK prediction
# # ============================================================
# import matplotlib.pyplot as plt

# print("\n===== Plot FRK prediction =====")

# # ----------------------------
# # 可調參數
# # ----------------------------
# plot_cell_idx = 0   # 要看的格點編號（0 ~ N-1）
# plot_h = 0          # 要看的 forecast step（0 ~ H-1）

# if not (0 <= plot_cell_idx < N):
#     raise ValueError(f"plot_cell_idx 必須介於 0 和 {N-1} 之間")
# if not (0 <= plot_h < H):
#     raise ValueError(f"plot_h 必須介於 0 和 {H-1} 之間")

# # ----------------------------
# # 準備資料
# # ----------------------------
# test_time_index = ts_df.index[cut_train:]

# # 真值 / DLinear / DLinear+FRK
# y_true_plot = y_test_true_combo[:, plot_cell_idx]
# y_dlinear_plot = trend_test_raw[:, plot_cell_idx]
# y_combo_plot = Yhat_test_combo[:, plot_cell_idx]

# # residual 真值與 residual 預測
# res_true_plot = y_true_plot - y_dlinear_plot
# res_pred_plot = Rhat_NH[plot_cell_idx, :]

# # ============================================================
# # 圖 1：單一格點時間序列比較
# # ============================================================
# plt.figure(figsize=(12, 5))
# plt.plot(test_time_index, y_true_plot, label="True")
# plt.plot(test_time_index, y_dlinear_plot, label="DLinear")
# plt.plot(test_time_index, y_combo_plot, label="DLinear + autoFRK")
# plt.title(
#     f"Forecast comparison at sampled cell #{plot_cell_idx}\n"
#     f"(lon={coords_sample[plot_cell_idx,0]:.2f}, lat={coords_sample[plot_cell_idx,1]:.2f})"
# )
# plt.xlabel("Time")
# plt.ylabel(var_name)
# plt.legend()
# plt.grid(True)
# plt.tight_layout()
# plt.show()

# # ============================================================
# # 圖 2：單一格點 residual 比較
# # ============================================================
# plt.figure(figsize=(12, 4))
# plt.plot(test_time_index, res_true_plot, label="True residual")
# plt.plot(test_time_index, res_pred_plot, label="autoFRK predicted residual")
# plt.title(
#     f"Residual forecast at sampled cell #{plot_cell_idx}\n"
#     f"(lon={coords_sample[plot_cell_idx,0]:.2f}, lat={coords_sample[plot_cell_idx,1]:.2f})"
# )
# plt.xlabel("Time")
# plt.ylabel("Residual")
# plt.legend()
# plt.grid(True)
# plt.tight_layout()
# plt.show()

# # ============================================================
# # 圖 3：指定 forecast step 的 autoFRK residual 空間分布
# # ============================================================
# plt.figure(figsize=(7, 5))
# sc = plt.scatter(
#     coords_sample[:, 0],
#     coords_sample[:, 1],
#     c=Rhat_NH[:, plot_h],
#     s=60
# )
# plt.colorbar(sc, label="Predicted residual")
# plt.title(f"autoFRK predicted residual field at forecast step {plot_h + 1}")
# plt.xlabel("Longitude")
# plt.ylabel("Latitude")
# plt.grid(True)
# plt.tight_layout()
# plt.show()

## 檢查 residual 的基本量級與空間結構強度

In [9]:
# Residual diagnostics: magnitude and covariance strength
import numpy as np
import pandas as pd

R_dlinear_train = R_hist_raw_TN.copy()   # (T_hist, N)
R_dlinear_test = y_test_true_combo - trend_test_raw
R_combo_test = y_test_true_combo - Yhat_test_combo

def safe_nanstd(x):
    return float(np.nanstd(np.asarray(x, dtype=np.float64)))

def cov_fro_norm(R):
    R = np.asarray(R, dtype=np.float64)
    col_mean = np.nanmean(R, axis=0, keepdims=True)
    Rc = R - col_mean
    valid_rows = ~np.isnan(Rc).any(axis=1)
    Rc = Rc[valid_rows]
    if Rc.shape[0] < 2:
        return np.nan
    Cov = np.cov(Rc, rowvar=False, ddof=1)
    return float(np.linalg.norm(Cov, ord="fro"))

diag_tbl_1 = pd.DataFrame({
    "Quantity": [
        "y_train std",
        "DLinear residual train std",
        "DLinear residual test std",
        "DLinear+FRK residual test std",
        "DLinear residual train covariance F-norm",
        "DLinear residual test covariance F-norm",
        "DLinear+FRK residual test covariance F-norm",
    ],
    "Value": [
        safe_nanstd(y_hist_raw_TN),
        safe_nanstd(R_dlinear_train),
        safe_nanstd(R_dlinear_test),
        safe_nanstd(R_combo_test),
        cov_fro_norm(R_dlinear_train),
        cov_fro_norm(R_dlinear_test),
        cov_fro_norm(R_combo_test),
    ]
})

print("\n===== Residual magnitude & covariance diagnostics =====")
print(diag_tbl_1.to_string(index=False))


===== Residual magnitude & covariance diagnostics =====
                                   Quantity      Value
                                y_train std  10.339348
                 DLinear residual train std   1.371540
                  DLinear residual test std   1.699785
              DLinear+FRK residual test std   1.673939
   DLinear residual train covariance F-norm  91.910436
    DLinear residual test covariance F-norm 122.064813
DLinear+FRK residual test covariance F-norm 133.584956


# 重複實驗10times

In [10]:
# # ============================================================
# # 10) Repeat full experiment 10 times
# #     - re-sample 100 US mainland cells each run
# #     - use fixed best DLinear params
# #     - compare DLinear(best) vs DLinear + autoFRK rolling
# # ============================================================
# import json
# from pathlib import Path

# import numpy as np
# import pandas as pd
# import torch

# from darts import TimeSeries
# from darts.models import DLinearModel
# from autoFRK import AutoFRK
# from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# print("\n===== Repeat full experiment =====")

# # ------------------------------------------------------------
# # Global settings
# # ------------------------------------------------------------
# N_RUNS = 10
# N_SAMPLE_TARGET = 100
# TRAIN_FRAC = 0.8

# USE_COVARIATES = False
# USE_REVIN = True
# BASE_RANDOM_STATE = 42

# PL_TRAINER_KWARGS = {
#     "enable_progress_bar": False,
#     "logger": False,
#     "enable_checkpointing": False,
# }

# BEST_PARAMS_PATH = Path("best_dlinear_params.json")
# REPEAT_RESULTS_PATH = Path("repeat_experiment_10runs_results.csv")
# REPEAT_SUMMARY_PATH = Path("repeat_experiment_10runs_summary.json")

# if not BEST_PARAMS_PATH.exists():
#     raise FileNotFoundError(f"找不到 {BEST_PARAMS_PATH.resolve()}，請先確認 best params 已存好。")

# with open(BEST_PARAMS_PATH, "r", encoding="utf-8") as f:
#     bp = json.load(f)

# print("Best params used:")
# for k, v in bp.items():
#     print(f"  {k} = {v}")

# # ------------------------------------------------------------
# # US mainland mask
# # ------------------------------------------------------------
# lon_all = gg[:, 0].astype(float)
# lat_all = gg[:, 1].astype(float)
# lon_all_180 = ((lon_all + 180) % 360) - 180

# lon_min, lon_max = -125, -66
# lat_min, lat_max = 24, 50

# mask_us_mainland = (
#     (lon_all_180 >= lon_min) & (lon_all_180 <= lon_max) &
#     (lat_all >= lat_min) & (lat_all <= lat_max)
# )

# idx_us_mainland = np.where(mask_us_mainland)[0]
# if len(idx_us_mainland) == 0:
#     raise ValueError("在設定的美國本土範圍內沒有格點。")

# print(f"US mainland cells available: {len(idx_us_mainland)}")

# # ------------------------------------------------------------
# # Helper functions
# # ------------------------------------------------------------
# def ts_to_2d(ts: TimeSeries) -> np.ndarray:
#     arr = np.asarray(ts.all_values(copy=False))
#     if arr.ndim == 3:
#         arr = arr[..., 0]
#     return arr

# def inverse_scale(x: np.ndarray, mean_vec: pd.Series, std_vec: pd.Series) -> np.ndarray:
#     return x * std_vec.values + mean_vec.values

# def run_one_experiment(run_id: int, sample_seed: int) -> dict:
#     print("\n" + "=" * 70)
#     print(f"Run {run_id + 1}/{N_RUNS} | sample_seed = {sample_seed}")

#     rng = np.random.default_rng(sample_seed)

#     n_sample = min(N_SAMPLE_TARGET, len(idx_us_mainland))
#     sample_idx_in_us = rng.choice(len(idx_us_mainland), size=n_sample, replace=False)
#     sample_idx_global = idx_us_mainland[sample_idx_in_us]

#     y_sample = y_all[sample_idx_global, :]
#     coords_sample = np.column_stack([
#         lon_all_180[sample_idx_global],
#         lat_all[sample_idx_global]
#     ]).astype(np.float64)

#     SAMPLE_SIZE = int(n_sample)
#     T = int(y_sample.shape[1])

#     ts_df = pd.DataFrame(
#         y_sample.T,
#         index=time_index,
#         columns=[f"cell_{i}" for i in range(SAMPLE_SIZE)]
#     ).astype("float32")

#     month_values = time_index.month.astype("float32")
#     month_df = pd.DataFrame({"month": month_values}, index=time_index)
#     month_ts = TimeSeries.from_dataframe(month_df.astype("float32"))

#     cut_train = int(T * TRAIN_FRAC)
#     split_time = ts_df.index[cut_train]

#     train_df = ts_df.iloc[:cut_train]
#     mean_vec = train_df.mean(axis=0)
#     std_vec = train_df.std(axis=0).replace(0.0, 1.0)

#     ts_df_scaled = (ts_df - mean_vec) / std_vec
#     ts_all_scaled = TimeSeries.from_dataframe(ts_df_scaled)

#     train_ts, test_ts = ts_all_scaled.split_before(split_time)

#     month_train, month_test = month_ts.split_before(split_time)
#     test_true_raw = ts_df.to_numpy(dtype=np.float32)[cut_train:, :]

#     T_train = len(train_ts)
#     T_test = len(test_ts)
#     N = SAMPLE_SIZE
#     H = T_test

#     print(f"Sample size = {N}, T_train = {T_train}, T_test = {T_test}")

#     # --------------------------------------------------------
#     # DLinear(best)
#     # --------------------------------------------------------
#     model_best_kwargs = dict(
#         input_chunk_length=int(bp["INPUT_CHUNK_LENGTH"]),
#         output_chunk_length=int(bp["OUTPUT_CHUNK_LENGTH"]),
#         kernel_size=int(bp["KERNEL_SIZE"]),
#         n_epochs=int(bp["N_EPOCHS"]),
#         random_state=int(bp.get("RANDOM_STATE", BASE_RANDOM_STATE)) + run_id,
#         use_reversible_instance_norm=bool(bp.get("USE_REVIN", USE_REVIN)),
#         batch_size=int(bp["BATCH_SIZE"]),
#         optimizer_kwargs={
#             "lr": float(bp["LR"]),
#             "weight_decay": float(bp["WEIGHT_DECAY"]),
#         },
#         const_init=bool(bp["CONST_INIT"]),
#         pl_trainer_kwargs=PL_TRAINER_KWARGS,
#         log_tensorboard=False,
#         save_checkpoints=False,
#     )

#     model_best = DLinearModel(**model_best_kwargs)

#     fit_kwargs_best = {"series": train_ts, "verbose": False}
#     pred_kwargs_best = {"n": T_test, "verbose": False, "show_warnings": False}

#     if bool(bp.get("USE_COVARIATES", USE_COVARIATES)):
#         fit_kwargs_best["past_covariates"] = month_train
#         fit_kwargs_best["future_covariates"] = month_train
#         pred_kwargs_best["past_covariates"] = month_ts
#         pred_kwargs_best["future_covariates"] = month_ts

#     model_best.fit(**fit_kwargs_best)
#     pred_best = model_best.predict(**pred_kwargs_best)

#     pred_best_scaled = ts_to_2d(pred_best)
#     pred_best_raw = inverse_scale(pred_best_scaled, mean_vec, std_vec)

#     y_test_true = test_true_raw.reshape(-1, SAMPLE_SIZE).astype(np.float64)
#     trend_test_raw = pred_best_raw.astype(np.float64)

#     mse_dlinear = mean_squared_error(y_test_true, trend_test_raw)
#     mae_dlinear = mean_absolute_error(y_test_true, trend_test_raw)
#     rmse_dlinear = float(np.sqrt(mse_dlinear))
#     r2_dlinear = r2_score(y_test_true, trend_test_raw)

#     # --------------------------------------------------------
#     # Build residual history on train
#     # --------------------------------------------------------
#     input_chunk_length_best = int(bp["INPUT_CHUNK_LENGTH"])

#     y_hist_raw_TN = ts_df.iloc[:cut_train, :].to_numpy(dtype=np.float64)
#     hist_index = ts_df.index[:cut_train]
#     T_hist = y_hist_raw_TN.shape[0]

#     pred_hist = model_best.historical_forecasts(
#         series=train_ts,
#         start=input_chunk_length_best,
#         forecast_horizon=1,
#         stride=1,
#         retrain=False,
#         last_points_only=True,
#         verbose=False,
#     )

#     if isinstance(pred_hist, list):
#         if len(pred_hist) == 0:
#             raise RuntimeError("historical_forecasts 回傳空 list，無法建立 residual history。")
#         pred_hist_ts = pred_hist[0]
#         for k in range(1, len(pred_hist)):
#             pred_hist_ts = pred_hist_ts.append(pred_hist[k])
#     else:
#         pred_hist_ts = pred_hist

#     pred_hist_scaled_TN = ts_to_2d(pred_hist_ts)
#     pred_time_index = pred_hist_ts.time_index

#     trend_hat_hist_raw_TN = np.full((T_hist, N), np.nan, dtype=np.float64)

#     pos = hist_index.get_indexer(pred_time_index)
#     valid = pos >= 0
#     pred_hist_scaled_TN = pred_hist_scaled_TN[valid]
#     pos = pos[valid]

#     pred_hist_raw_TN = inverse_scale(pred_hist_scaled_TN, mean_vec, std_vec)
#     trend_hat_hist_raw_TN[pos] = pred_hist_raw_TN

#     R_hist_raw_TN = y_hist_raw_TN - trend_hat_hist_raw_TN
#     R_hist_raw_NT = R_hist_raw_TN.T

#     nan_ratio = float(np.isnan(R_hist_raw_NT).mean())

#     valid_cols = ~np.isnan(R_hist_raw_NT).any(axis=0)
#     R_hist_valid_NT = R_hist_raw_NT[:, valid_cols]

#     if R_hist_valid_NT.shape[1] < 2:
#         raise RuntimeError("有效 residual 歷史長度不足，無法做 rolling latent forecast。")

#     # --------------------------------------------------------
#     # autoFRK rolling residual forecast
#     # --------------------------------------------------------
#     loc = torch.from_numpy(coords_sample).to(dtype=torch.float64, device="cpu")
#     afrk = AutoFRK(dtype=torch.float64, device="cpu")

#     Rhat_roll_NH = np.zeros((N, H), dtype=np.float64)
#     K_list = []

#     R_hist_roll_NT = R_hist_valid_NT.copy()

#     for h in range(H):
#         data_hist = torch.from_numpy(R_hist_roll_NT).to(dtype=torch.float64)

#         result_h = afrk.forward(
#             data=data_hist,
#             loc=loc,
#             method="EM",
#             maxit=50,
#             tolerance=1e-6,
#             n_neighbor=3,
#             tps_method="spherical_fast",
#         )

#         w_hist = result_h["w"]
#         w_hist_np = w_hist.detach().cpu().numpy()
#         K, T_latent = w_hist_np.shape
#         K_list.append(int(K))

#         if T_latent < 2:
#             raise RuntimeError(f"第 {h + 1} 步 latent 歷史長度不足，無法建立 VAR(1)。")

#         W0 = w_hist_np[:, :-1]
#         W1 = w_hist_np[:, 1:]
#         A = (W1 @ W0.T) @ np.linalg.pinv(W0 @ W0.T)

#         w_last = w_hist_np[:, -1]
#         w_next = (A @ w_last).reshape(K, 1)

#         obj_next = dict(result_h)
#         obj_next["w"] = torch.from_numpy(w_next).to(dtype=torch.float64)

#         pred_res_h = afrk.predict(
#             obj=obj_next,
#             newloc=loc,
#             se_report=False,
#             tps_method="spherical_fast",
#         )

#         rhat_h = pred_res_h["pred.value"].detach().cpu().numpy()

#         if rhat_h.ndim == 2 and rhat_h.shape == (N, 1):
#             rhat_h = rhat_h[:, 0]
#         elif rhat_h.ndim == 1 and rhat_h.shape[0] == N:
#             pass
#         else:
#             raise ValueError(f"第 {h + 1} 步 residual prediction shape 異常: {rhat_h.shape}")

#         Rhat_roll_NH[:, h] = rhat_h

#         y_true_h = y_test_true[h, :]
#         trend_h = trend_test_raw[h, :]
#         r_true_h = y_true_h - trend_h

#         R_hist_roll_NT = np.column_stack([R_hist_roll_NT, r_true_h])

#     Yhat_test_combo = trend_test_raw + Rhat_roll_NH.T

#     mse_combo = mean_squared_error(y_test_true, Yhat_test_combo)
#     mae_combo = mean_absolute_error(y_test_true, Yhat_test_combo)
#     rmse_combo = float(np.sqrt(mse_combo))
#     r2_combo = r2_score(y_test_true, Yhat_test_combo)

#     print(f"DLinear RMSE            = {rmse_dlinear:.6f}")
#     print(f"DLinear + autoFRK RMSE  = {rmse_combo:.6f}")

#     return {
#         "run": int(run_id + 1),
#         "sample_seed": int(sample_seed),
#         "n_sample": int(N),
#         "t_train": int(T_train),
#         "t_test": int(T_test),
#         "dlinear_rmse": float(rmse_dlinear),
#         "dlinear_mse": float(mse_dlinear),
#         "dlinear_mae": float(mae_dlinear),
#         "dlinear_r2": float(r2_dlinear),
#         "frk_rmse": float(rmse_combo),
#         "frk_mse": float(mse_combo),
#         "frk_mae": float(mae_combo),
#         "frk_r2": float(r2_combo),
#         "rmse_improve_pct": float((rmse_dlinear - rmse_combo) / rmse_dlinear * 100.0),
#         "nan_ratio": float(nan_ratio),
#         "k_first_step": int(K_list[0]) if len(K_list) > 0 else None,
#         "k_last_step": int(K_list[-1]) if len(K_list) > 0 else None,
#     }

# # ------------------------------------------------------------
# # Run all experiments
# # ------------------------------------------------------------
# results = []

# for run_id in range(N_RUNS):
#     sample_seed = 42 + run_id
#     out = run_one_experiment(run_id=run_id, sample_seed=sample_seed)
#     results.append(out)

# results_df = pd.DataFrame(results)
# results_df.to_csv(REPEAT_RESULTS_PATH, index=False, encoding="utf-8-sig")

# print("\n===== Per-run results =====")
# print(results_df.to_string(index=False))

# # ------------------------------------------------------------
# # Summary
# # ------------------------------------------------------------
# summary = {
#     "n_runs": int(N_RUNS),
#     "dlinear_rmse_mean": float(results_df["dlinear_rmse"].mean()),
#     "dlinear_rmse_sd": float(results_df["dlinear_rmse"].std(ddof=1)),
#     "frk_rmse_mean": float(results_df["frk_rmse"].mean()),
#     "frk_rmse_sd": float(results_df["frk_rmse"].std(ddof=1)),
#     "dlinear_mse_mean": float(results_df["dlinear_mse"].mean()),
#     "dlinear_mse_sd": float(results_df["dlinear_mse"].std(ddof=1)),
#     "frk_mse_mean": float(results_df["frk_mse"].mean()),
#     "frk_mse_sd": float(results_df["frk_mse"].std(ddof=1)),
#     "dlinear_mae_mean": float(results_df["dlinear_mae"].mean()),
#     "dlinear_mae_sd": float(results_df["dlinear_mae"].std(ddof=1)),
#     "frk_mae_mean": float(results_df["frk_mae"].mean()),
#     "frk_mae_sd": float(results_df["frk_mae"].std(ddof=1)),
#     "dlinear_r2_mean": float(results_df["dlinear_r2"].mean()),
#     "dlinear_r2_sd": float(results_df["dlinear_r2"].std(ddof=1)),
#     "frk_r2_mean": float(results_df["frk_r2"].mean()),
#     "frk_r2_sd": float(results_df["frk_r2"].std(ddof=1)),
#     "rmse_improve_pct_mean": float(results_df["rmse_improve_pct"].mean()),
#     "rmse_improve_pct_sd": float(results_df["rmse_improve_pct"].std(ddof=1)),
# }

# with open(REPEAT_SUMMARY_PATH, "w", encoding="utf-8") as f:
#     json.dump(summary, f, ensure_ascii=False, indent=2)

# summary_table = pd.DataFrame({
#     "Model": ["DLinear(best)", "DLinear + autoFRK rolling"],
#     "RMSE": [
#         f"{summary['dlinear_rmse_mean']:.6f} ± {summary['dlinear_rmse_sd']:.6f}",
#         f"{summary['frk_rmse_mean']:.6f} ± {summary['frk_rmse_sd']:.6f}",
#     ],
#     "MSE": [
#         f"{summary['dlinear_mse_mean']:.6f} ± {summary['dlinear_mse_sd']:.6f}",
#         f"{summary['frk_mse_mean']:.6f} ± {summary['frk_mse_sd']:.6f}",
#     ],
#     "MAE": [
#         f"{summary['dlinear_mae_mean']:.6f} ± {summary['dlinear_mae_sd']:.6f}",
#         f"{summary['frk_mae_mean']:.6f} ± {summary['frk_mae_sd']:.6f}",
#     ],
#     "R2": [
#         f"{summary['dlinear_r2_mean']:.6f} ± {summary['dlinear_r2_sd']:.6f}",
#         f"{summary['frk_r2_mean']:.6f} ± {summary['frk_r2_sd']:.6f}",
#     ],
# })

# print("\n===== Summary over 10 runs =====")
# print(summary_table.to_string(index=False))
# print(f"\nAverage RMSE improvement (%) = {summary['rmse_improve_pct_mean']:.6f} ± {summary['rmse_improve_pct_sd']:.6f}")

# print(f"\nSaved per-run results: {REPEAT_RESULTS_PATH.resolve()}")
# print(f"Saved summary: {REPEAT_SUMMARY_PATH.resolve()}")

# autoFRK (500)

## 確認變數

In [11]:
# print(sorted(globals().keys()))

# target_vars = [
#     "coords_sample",
#     "coords_500_raw",
#     "y_500_raw",
#     "pred_best_raw",
#     "R_hist_valid_NT",
#     "test_true_raw",
#     "T_test",
#     "cut_val",
#     "time_index",
# ]

# for v in target_vars:
#     print(v, "✅" if v in globals() else "❌")

In [12]:
# candidate_vars = [
#     "y_all",
#     "y_us",
#     "coords_us",
#     "lat_all",
#     "lon_all",
#     "lat_us_sample",
#     "lon_us_sample",
#     "lat_grid",
#     "lon_grid",
# ]

# print("===== candidate variable shapes =====")
# for name in candidate_vars:
#     if name in globals():
#         obj = globals()[name]
#         try:
#             print(f"{name}: shape = {np.shape(obj)}, type = {type(obj)}")
#         except Exception:
#             print(f"{name}: type = {type(obj)}")
#     else:
#         print(f"{name}: NOT FOUND")

## 設定變數

In [13]:
# # ============================================================
# # Build 500-site target set from US pool
# # ============================================================
# import numpy as np

# N_TARGET_500 = 500
# TARGET_SEED_500 = 42
# EXCLUDE_OBS_100_FROM_TARGET_500 = True

# required_names = ["coords_us", "y_us", "sample_idx_in_us"]
# for name in required_names:
#     if name not in globals():
#         raise RuntimeError(f"缺少必要變數: {name}")

# coords_us_arr = np.asarray(coords_us, dtype=np.float64)
# y_us_arr = np.asarray(y_us, dtype=np.float64)
# sample_idx_in_us_arr = np.asarray(sample_idx_in_us, dtype=int).reshape(-1)

# if coords_us_arr.ndim != 2 or coords_us_arr.shape[1] != 2:
#     raise ValueError(f"coords_us.shape = {coords_us_arr.shape}，應為 (n_sites, 2)")

# if y_us_arr.ndim != 2:
#     raise ValueError(f"y_us.shape = {y_us_arr.shape}，應為 (n_sites, T)")

# if coords_us_arr.shape[0] != y_us_arr.shape[0]:
#     raise ValueError(
#         f"coords_us.shape[0] = {coords_us_arr.shape[0]}，"
#         f"但 y_us.shape[0] = {y_us_arr.shape[0]}，兩者不一致"
#     )

# n_us_sites = coords_us_arr.shape[0]

# if EXCLUDE_OBS_100_FROM_TARGET_500:
#     candidate_idx = np.setdiff1d(np.arange(n_us_sites), sample_idx_in_us_arr)
# else:
#     candidate_idx = np.arange(n_us_sites)

# if candidate_idx.shape[0] < N_TARGET_500:
#     raise ValueError(
#         f"可選格點數只有 {candidate_idx.shape[0]}，不足以抽 {N_TARGET_500} 個"
#     )

# rng = np.random.default_rng(TARGET_SEED_500)
# target_idx_in_us = np.sort(rng.choice(candidate_idx, size=N_TARGET_500, replace=False))

# coords_500_raw = coords_us_arr[target_idx_in_us, :]
# y_500_raw = y_us_arr[target_idx_in_us, :]

# print("===== 500-site target set built =====")
# print("coords_500_raw shape =", coords_500_raw.shape)
# print("y_500_raw shape      =", y_500_raw.shape)
# print("n_us_sites           =", n_us_sites)
# print("n_obs_100            =", len(sample_idx_in_us_arr))
# print("n_target_500         =", len(target_idx_in_us))
# print("exclude_obs_100      =", EXCLUDE_OBS_100_FROM_TARGET_500)

# overlap_n = np.intersect1d(sample_idx_in_us_arr, target_idx_in_us).shape[0]
# print("overlap with obs 100 =", overlap_n)

# if coords_500_raw.shape != (500, 2):
#     raise ValueError(f"coords_500_raw.shape = {coords_500_raw.shape}，應為 (500, 2)")

# if y_500_raw.shape[0] != 500:
#     raise ValueError(f"y_500_raw.shape[0] = {y_500_raw.shape[0]}，應為 500")

# if "time_index" in globals():
#     if y_500_raw.shape[1] != len(time_index):
#         raise ValueError(
#             f"y_500_raw.shape[1] = {y_500_raw.shape[1]}，"
#             f"但 len(time_index) = {len(time_index)}"
#         )

## main

In [16]:
# ============================================================
# 10) DLinear(100) + autoFRK rolling residual forecast -> 500 sites
#    Reuse 100-site trained DLinear + residual history
# ============================================================
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from autoFRK import AutoFRK
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("\n===== DLinear(100) + autoFRK rolling residual forecast -> 500 sites =====")

PRED500_METRICS_PATH = Path("dlinear_100sites_to_500sites_autofrk_metrics.json")

# ------------------------------------------------------------
# 0) 500-site target settings
# ------------------------------------------------------------
AUTO_BUILD_500_TARGET = True
N_TARGET_500 = 500
TARGET_SEED_500 = 123
EXCLUDE_OBS_100_FROM_TARGET_500 = True

# 如果你已經手動建好 500-site target，改成 False
COORDS_500_VAR_NAME = "coords_500_raw"
Y_500_VAR_NAME = "y_500_raw"


def _json_dump(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def _cleanup_torch_cache() -> None:
    try:
        import torch as _torch
        if _torch.cuda.is_available():
            _torch.cuda.empty_cache()
    except Exception:
        pass


def _to_2d_np(x):
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    arr = np.asarray(x)
    if arr.ndim == 3 and arr.shape[-1] == 1:
        arr = arr[..., 0]
    return arr.astype(np.float64, copy=False)


# ------------------------------------------------------------
# 1) Check required objects from 100-site run
# ------------------------------------------------------------
required_names = [
    "coords_sample",
    "model_best",
    "pred_best_raw",
    "R_hist_valid_NT",
    "test_true_raw",
    "T_test",
    "cut_val",
    "time_index",
]

for name in required_names:
    if name not in globals():
        raise RuntimeError(f"缺少必要物件: {name}。請先完整跑完 100-site DLinear + autoFRK chunk。")

coords_obs = np.asarray(coords_sample, dtype=np.float64)
loc_obs = torch.from_numpy(coords_obs).to(dtype=torch.float64, device="cpu")

N_obs = coords_obs.shape[0]
H = int(T_test)

if N_obs != 100:
    raise ValueError(f"這一段是建立在 100-site 結果上，目前 N_obs = {N_obs}，不是 100。")

if coords_obs.ndim != 2 or coords_obs.shape[1] != 2:
    raise ValueError(f"coords_sample.shape = {coords_obs.shape}，應為 (100, 2)")

if R_hist_valid_NT.shape[0] != 100:
    raise ValueError(f"R_hist_valid_NT.shape[0] = {R_hist_valid_NT.shape[0]}，應為 100")

# ------------------------------------------------------------
# 2) Build / load 500-site target set
# ------------------------------------------------------------
if AUTO_BUILD_500_TARGET:
    build_required = ["coords_us", "y_us", "sample_idx_in_us"]
    for name in build_required:
        if name not in globals():
            raise RuntimeError(
                f"缺少必要物件: {name}。AUTO_BUILD_500_TARGET=True 時，"
                f"需要 coords_us / y_us / sample_idx_in_us。"
            )

    coords_us_arr = np.asarray(coords_us, dtype=np.float64)
    y_us_arr = np.asarray(y_us, dtype=np.float64)
    sample_idx_in_us_arr = np.asarray(sample_idx_in_us, dtype=int).reshape(-1)

    if coords_us_arr.ndim != 2 or coords_us_arr.shape[1] != 2:
        raise ValueError(f"coords_us.shape = {coords_us_arr.shape}，應為 (n_sites, 2)")

    if y_us_arr.ndim != 2:
        raise ValueError(f"y_us.shape = {y_us_arr.shape}，應為 (n_sites, T)")

    if coords_us_arr.shape[0] != y_us_arr.shape[0]:
        raise ValueError(
            f"coords_us.shape[0] = {coords_us_arr.shape[0]}，"
            f"但 y_us.shape[0] = {y_us_arr.shape[0]}，兩者不一致"
        )

    n_us_sites = coords_us_arr.shape[0]

    if EXCLUDE_OBS_100_FROM_TARGET_500:
        candidate_idx = np.setdiff1d(np.arange(n_us_sites), sample_idx_in_us_arr)
    else:
        candidate_idx = np.arange(n_us_sites)

    if candidate_idx.shape[0] < N_TARGET_500:
        raise ValueError(
            f"可選格點數只有 {candidate_idx.shape[0]}，不足以抽 {N_TARGET_500} 個"
        )

    rng = np.random.default_rng(TARGET_SEED_500)
    target_idx_in_us = np.sort(
        rng.choice(candidate_idx, size=N_TARGET_500, replace=False)
    )

    coords_500_raw = coords_us_arr[target_idx_in_us, :]
    y_500_raw = y_us_arr[target_idx_in_us, :]

    overlap_n = np.intersect1d(sample_idx_in_us_arr, target_idx_in_us).shape[0]

    print("\n===== Built fresh 500-site target set =====")
    print("coords_500_raw shape =", coords_500_raw.shape)
    print("y_500_raw shape      =", y_500_raw.shape)
    print("n_obs_100            =", len(sample_idx_in_us_arr))
    print("n_target_500         =", len(target_idx_in_us))
    print("exclude_obs_100      =", EXCLUDE_OBS_100_FROM_TARGET_500)
    print("overlap with obs 100 =", overlap_n)

if COORDS_500_VAR_NAME not in globals():
    raise RuntimeError(f"找不到 500 格點座標變數: {COORDS_500_VAR_NAME}")

if Y_500_VAR_NAME not in globals():
    raise RuntimeError(f"找不到 500 格點完整資料變數: {Y_500_VAR_NAME}")

coords_full = np.asarray(globals()[COORDS_500_VAR_NAME], dtype=np.float64)
y_full = np.asarray(globals()[Y_500_VAR_NAME], dtype=np.float64)
loc_full = torch.from_numpy(coords_full).to(dtype=torch.float64, device="cpu")

N_full = coords_full.shape[0]

if coords_full.ndim != 2 or coords_full.shape[1] != 2:
    raise ValueError(f"{COORDS_500_VAR_NAME}.shape = {coords_full.shape}，應為 (500, 2)")

if N_full != 500:
    raise ValueError(f"{COORDS_500_VAR_NAME} 的格點數 = {N_full}，這個 chunk 預期是 500")

if y_full.ndim != 2:
    raise ValueError(f"{Y_500_VAR_NAME}.shape = {y_full.shape}，應為 (500, T)")

if y_full.shape[0] != 500:
    raise ValueError(f"{Y_500_VAR_NAME}.shape[0] = {y_full.shape[0]}，應為 500")

if y_full.shape[1] != len(time_index):
    raise ValueError(
        f"{Y_500_VAR_NAME}.shape[1] = {y_full.shape[1]}，但 len(time_index) = {len(time_index)}"
    )

print(f"\nObserved sites: {N_obs}")
print(f"Target sites: {N_full}")
print(f"Horizon H: {H}")
print(f"Using 500-site coords: {COORDS_500_VAR_NAME}, shape = {coords_full.shape}")
print(f"Using 500-site data:   {Y_500_VAR_NAME}, shape = {y_full.shape}")

# ------------------------------------------------------------
# 3) 100-site test trend
# ------------------------------------------------------------
trend_test_obs_TN = np.asarray(pred_best_raw, dtype=np.float64)

if trend_test_obs_TN.shape != (H, N_obs):
    raise ValueError(
        f"pred_best_raw.shape = {trend_test_obs_TN.shape}，應為 {(H, N_obs)}"
    )

y_test_true_obs_TN = test_true_raw.reshape(H, N_obs).astype(np.float64)
r_test_true_obs_TN = y_test_true_obs_TN - trend_test_obs_TN

# ------------------------------------------------------------
# 4) 500-site true test data
# ------------------------------------------------------------
y_test_true_full_TN = y_full[:, cut_val:].T.astype(np.float64)

if y_test_true_full_TN.shape != (H, N_full):
    raise ValueError(
        f"500-site test true shape = {y_test_true_full_TN.shape}，應為 {(H, N_full)}"
    )

# ------------------------------------------------------------
# 5) DLinear trend: 100 sites -> 500 sites
# ------------------------------------------------------------
trend_obs_NH = trend_test_obs_TN.T

afrk_trend = AutoFRK(dtype=torch.float64, device="cpu")

trend_result = afrk_trend.forward(
    data=torch.from_numpy(trend_obs_NH).to(dtype=torch.float64),
    loc=loc_obs,
    method="EM",
    maxit=50,
    tolerance=1e-6,
    n_neighbor=3,
    tps_method="spherical_fast",
)

trend_pred_full = afrk_trend.predict(
    obj=trend_result,
    newloc=loc_full,
    se_report=False,
    tps_method="spherical_fast",
)

trend_test_full_arr = _to_2d_np(trend_pred_full["pred.value"])

if trend_test_full_arr.shape == (N_full, H):
    trend_test_full_TN = trend_test_full_arr.T
elif trend_test_full_arr.shape == (H, N_full):
    trend_test_full_TN = trend_test_full_arr
else:
    raise ValueError(
        f"500-site trend prediction shape 異常: {trend_test_full_arr.shape}"
    )

# ------------------------------------------------------------
# 6) Rolling residual forecast: 100 sites -> 500 sites
# ------------------------------------------------------------
afrk_resid = AutoFRK(dtype=torch.float64, device="cpu")

Rhat_roll_full_NH = np.zeros((N_full, H), dtype=np.float64)
K_list = []

R_hist_roll_obs_NT = R_hist_valid_NT.copy()

for h in range(H):
    data_hist = torch.from_numpy(R_hist_roll_obs_NT).to(dtype=torch.float64)

    result_h = afrk_resid.forward(
        data=data_hist,
        loc=loc_obs,
        method="EM",
        maxit=50,
        tolerance=1e-6,
        n_neighbor=3,
        tps_method="spherical_fast",
    )

    w_hist = result_h["w"]
    w_hist_np = w_hist.detach().cpu().numpy()
    K, T_latent = w_hist_np.shape
    K_list.append(int(K))

    if T_latent < 2:
        raise RuntimeError(f"第 {h+1} 步 latent 歷史長度不足，無法建立 VAR(1)。")

    W0 = w_hist_np[:, :-1]
    W1 = w_hist_np[:, 1:]
    A = (W1 @ W0.T) @ np.linalg.pinv(W0 @ W0.T)

    w_last = w_hist_np[:, -1]
    w_next = (A @ w_last).reshape(K, 1)

    obj_next = dict(result_h)
    obj_next["w"] = torch.from_numpy(w_next).to(dtype=torch.float64)

    pred_res_h = afrk_resid.predict(
        obj=obj_next,
        newloc=loc_full,
        se_report=False,
        tps_method="spherical_fast",
    )

    rhat_h = _to_2d_np(pred_res_h["pred.value"])

    if rhat_h.ndim == 2 and rhat_h.shape == (N_full, 1):
        rhat_h = rhat_h[:, 0]
    elif rhat_h.ndim == 1 and rhat_h.shape[0] == N_full:
        pass
    else:
        raise ValueError(f"第 {h+1} 步 residual prediction shape 異常: {rhat_h.shape}")

    Rhat_roll_full_NH[:, h] = rhat_h

    y_true_h = y_test_true_obs_TN[h, :]
    trend_h = trend_test_obs_TN[h, :]
    r_true_h = y_true_h - trend_h

    R_hist_roll_obs_NT = np.column_stack([R_hist_roll_obs_NT, r_true_h])

# ------------------------------------------------------------
# 7) Final combined forecast on 500 sites
# ------------------------------------------------------------
Yhat_test_full_TN = trend_test_full_TN + Rhat_roll_full_NH.T

mse_trend500 = mean_squared_error(y_test_true_full_TN, trend_test_full_TN)
mae_trend500 = mean_absolute_error(y_test_true_full_TN, trend_test_full_TN)
rmse_trend500 = float(np.sqrt(mse_trend500))
r2_trend500 = r2_score(y_test_true_full_TN, trend_test_full_TN)

mse_combo500 = mean_squared_error(y_test_true_full_TN, Yhat_test_full_TN)
mae_combo500 = mean_absolute_error(y_test_true_full_TN, Yhat_test_full_TN)
rmse_combo500 = float(np.sqrt(mse_combo500))
r2_combo500 = r2_score(y_test_true_full_TN, Yhat_test_full_TN)

metrics_500 = pd.DataFrame({
    "Model": [
        "DLinear(100) trend -> FRK spatial 500",
        "DLinear(100) + autoFRK residual -> 500",
    ],
    "Split": ["TEST_500", "TEST_500"],
    "RMSE": [rmse_trend500, rmse_combo500],
    "MSE": [float(mse_trend500), float(mse_combo500)],
    "MAE": [float(mae_trend500), float(mae_combo500)],
    "R2": [float(r2_trend500), float(r2_combo500)],
})

print("\n===== 500-site forecast results =====")
print(metrics_500.to_string(index=False))

payload_500 = {
    "trend_only_500": {
        "RMSE": float(rmse_trend500),
        "MSE": float(mse_trend500),
        "MAE": float(mae_trend500),
        "R2": float(r2_trend500),
    },
    "trend_plus_residual_500": {
        "RMSE": float(rmse_combo500),
        "MSE": float(mse_combo500),
        "MAE": float(mae_combo500),
        "R2": float(r2_combo500),
    },
    "obs_sites": int(N_obs),
    "target_sites": int(N_full),
    "H": int(H),
    "coords_500_var_name": COORDS_500_VAR_NAME,
    "y_500_var_name": Y_500_VAR_NAME,
    "auto_build_500_target": bool(AUTO_BUILD_500_TARGET),
    "target_seed_500": int(TARGET_SEED_500),
    "exclude_obs_100_from_target_500": bool(EXCLUDE_OBS_100_FROM_TARGET_500),
    "K_first_step": int(K_list[0]) if len(K_list) > 0 else None,
    "K_last_step": int(K_list[-1]) if len(K_list) > 0 else None,
    "split_ratio": {
        "train": 0.7,
        "val": 0.15,
        "test": 0.15,
    },
}

_json_dump(payload_500, PRED500_METRICS_PATH)
print(f"\nSaved 500-site metrics: {PRED500_METRICS_PATH.resolve()}")

del afrk_trend, afrk_resid
gc.collect()
_cleanup_torch_cache()

2026-03-25 21:22:56 - autoFRK.utils.logger - INFO: Successfully using device "cpu".
2026-03-25 21:22:56 - autoFRK.utils.logger - INFO: Calculate TPS with spherical_fast.



===== DLinear(100) + autoFRK rolling residual forecast -> 500 sites =====

===== Built fresh 500-site target set =====
coords_500_raw shape = (500, 2)
y_500_raw shape      = (500, 548)
n_obs_100            = 100
n_target_500         = 500
exclude_obs_100      = True
overlap with obs 100 = 0

Observed sites: 100
Target sites: 500
Horizon H: 83
Using 500-site coords: coords_500_raw, shape = (500, 2)
Using 500-site data:   y_500_raw, shape = (500, 548)


2026-03-25 21:22:56 - autoFRK.utils.logger - INFO: Successfully using device "cpu".
2026-03-25 21:22:56 - autoFRK.utils.logger - INFO: Calculate TPS with spherical_fast.
2026-03-25 21:22:57 - autoFRK.utils.logger - INFO: Successfully using device "cpu".
2026-03-25 21:22:57 - autoFRK.utils.logger - INFO: Calculate TPS with spherical_fast.
2026-03-25 21:22:57 - autoFRK.utils.logger - INFO: Successfully using device "cpu".
2026-03-25 21:22:57 - autoFRK.utils.logger - INFO: Calculate TPS with spherical_fast.
2026-03-25 21:22:57 - autoFRK.utils.logger - INFO: Successfully using device "cpu".
2026-03-25 21:22:57 - autoFRK.utils.logger - INFO: Calculate TPS with spherical_fast.
2026-03-25 21:22:57 - autoFRK.utils.logger - INFO: Successfully using device "cpu".
2026-03-25 21:22:57 - autoFRK.utils.logger - INFO: Calculate TPS with spherical_fast.
2026-03-25 21:22:58 - autoFRK.utils.logger - INFO: Successfully using device "cpu".
2026-03-25 21:22:58 - autoFRK.utils.logger - INFO: Calculate TPS w


===== 500-site forecast results =====
                                 Model    Split     RMSE      MSE      MAE       R2
 DLinear(100) trend -> FRK spatial 500 TEST_500 2.781475 7.736601 2.089679 0.692713
DLinear(100) + autoFRK residual -> 500 TEST_500 2.757602 7.604367 2.041844 0.744391

Saved 500-site metrics: /home/jundian/Research-Project/project/dlinear_100sites_to_500sites_autofrk_metrics.json


## diagnosed

In [17]:
print("residual std (train):", np.std(R_hist_valid_NT))
print("trend std:", np.std(pred_best_raw))

residual std (train): 1.3715396888159452
trend std: 10.257316900815164


# finish